In [1]:
# 1. Cài đặt các thư viện Python cần thiết
!pip install -q selenium webdriver-manager bs4 requests pandas urllib3

# 2. Thêm key và repository chính thức của Google Chrome
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'

# 3. Cập nhật apt và cài đặt Google Chrome Stable
!apt-get update -y -q
!apt-get install -y google-chrome-stable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 26.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,220 B]
Hit:6 http://archive.ubu

In [ ]:
import time
import re
import os
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- CẤU HÌNH HỆ THỐNG ---
TARGET_YEARS = [2023, 2024, 2025] 

CATEGORIES_DATA = {
    "thoi su": {
        "but bi": "https://tuoitre.vn/thoi-su/but-bi.htm",
        "xa hoi": "https://tuoitre.vn/thoi-su/xa-hoi.htm",
        "phong su": "https://tuoitre.vn/thoi-su/phong-su.htm",
        "binh luan": "https://tuoitre.vn/thoi-su/binh-luan.htm"
    },
    "the gioi": {
        "binh luan": "https://tuoitre.vn/the-gioi/binh-luan.htm",
        "kieu bao": "https://tuoitre.vn/the-gioi/kieu-bao.htm",
        "muon mau": "https://tuoitre.vn/the-gioi/muon-mau.htm",
        "ho so": "https://tuoitre.vn/the-gioi/ho-so.htm",
        "bau cu my 2024": "https://tuoitre.vn/bau-cu-tong-thong-my-e587.htm"
    },
    "phap luat": {
        "chuyen phap đinh": "https://tuoitre.vn/phap-luat/chuyen-phap-dinh.htm",
        "tu van": "https://tuoitre.vn/phap-luat/tu-van.htm",
        "phap ly": "https://tuoitre.vn/phap-luat/phap-ly.htm"
    },
    "kinh doanh": {
        "tai chinh": "https://tuoitre.vn/kinh-doanh/tai-chinh.htm",
        "doanh nghiep": "https://tuoitre.vn/kinh-doanh/doanh-nghiep.htm",
        "mua sam": "https://tuoitre.vn/kinh-doanh/mua-sam.htm",
        "chung khoan": "https://tuoitre.vn/chung-khoan.htm",
        "gia vang hom nay": "https://tuoitre.vn/gia-vang-e592.htm"
    },
    "cong nghe": {
        "thiet bi": "https://tuoitre.vn/cong-nghe/thiet-bi.htm",
        "chuyen đoi so": "https://tuoitre.vn/cong-nghe/chuyen-doi-so.htm",
        "cau noi": "https://tuoitre.vn/cong-nghe/cau-noi.htm",
        "nhip song so": "https://tuoitre.vn/cong-nghe/nhip-song-so.htm",
        "trai nghiem": "https://tuoitre.vn/cong-nghe/trai-nghiem.htm",
        "thi truong": "https://tuoitre.vn/cong-nghe/thi-truong.htm",
        "thu thuat": "https://tuoitre.vn/cong-nghe/thu-thuat.htm",
        "gia đinh so": "https://tuoitre.vn/cong-nghe/gia-dinh-so.htm",
        "nhip cau": "https://tuoitre.vn/cong-nghe/nhip-cau.htm"
    },
    "xe": {
        "tin tuc": "https://tuoitre.vn/xe/tin-tuc.htm",
        "tu van mua xe": "https://tuoitre.vn/xe/tu-van-mua-xe.htm",
        "đanh gia xe": "https://tuoitre.vn/xe/danh-gia-xe.htm",
        "thi truong xe": "https://tuoitre.vn/xe/thi-truong-xe.htm",
        "hoi đap": "https://tuoitre.vn/xe/hoi-dap.htm",
        "ky thuat": "https://tuoitre.vn/xe/ky-thuat.htm",
        "an toan giao thong": "https://tuoitre.vn/xe/an-toan-giao-thong.htm"
    },
    "du lich": {
        "co hoi du lich": "https://tuoitre.vn/du-lich/co-hoi-du-lich.htm",
        "đi choi": "https://tuoitre.vn/du-lich/di-choi.htm",
        "mach ban": "https://tuoitre.vn/du-lich/mach-ban.htm",
        "que huong": "https://tuoitre.vn/du-lich/que-huong.htm"
    },
    "giai tri": {
        "nghe gi hom nay": "https://tuoitre.vn/giai-tri/nghe-gi-hom-nay.htm",
        "am nhac": "https://tuoitre.vn/giai-tri/am-nhac.htm",
        "đien anh": "https://tuoitre.vn/giai-tri/dien-anh.htm",
        "tv show": "https://tuoitre.vn/giai-tri/tv-show.htm",
        "thoi trang": "https://tuoitre.vn/giai-tri/thoi-trang.htm",
        "hau truong": "https://tuoitre.vn/giai-tri/hau-truong.htm"
    },
    "the thao": {
        "bong đa": "https://tuoitre.vn/the-thao/bong-da.htm",
        "bong chuyen": "https://tuoitre.vn/the-thao/bong-chuyen.htm",
        "vo thuat": "https://tuoitre.vn/the-thao/vo-thuat.htm",
        "cac mon khac": "https://tuoitre.vn/the-thao/cac-mon-khac.htm",
        "khoe 360°": "https://tuoitre.vn/the-thao/khoe-360.htm",
        "lich thi đau": "https://tuoitre.vn/the-thao/lich-thi-dau.htm"
    },
    "giao duc": {
        "tuyen sinh": "https://tuoitre.vn/giao-duc/tuyen-sinh.htm",
        "nhip song hoc đuong": "https://tuoitre.vn/giao-duc/nhip-song-hoc-duong.htm",
        "chan dung nha giao": "https://tuoitre.vn/giao-duc/chan-dung-nha-giao.htm",
        "du hoc": "https://tuoitre.vn/giao-duc/du-hoc.htm",
        "cau chuyen giao duc": "https://tuoitre.vn/giao-duc/cau-chuyen-giao-duc.htm",
        "điem thi": "https://tuoitre.vnhttps://tuoitre.vn/diem-thi.htm"
    },
    "nha đat": {
        "hoi đap": "https://tuoitre.vn/nha-dat/hoi-dap.htm",
        "thi truong": "https://tuoitre.vn/nha-dat/thi-truong.htm",
        "chinh sach": "https://tuoitre.vn/nha-dat/chinh-sach.htm",
        "du an": "https://tuoitre.vn/nha-dat/du-an.htm"
    },
    "suc khoe": {
        "dinh duong": "https://tuoitre.vn/suc-khoe/dinh-duong.htm",
        "me & be": "https://tuoitre.vn/suc-khoe/me-va-be.htm",
        "gioi tinh": "https://tuoitre.vn/suc-khoe/gioi-tinh.htm",
        "phong mach": "https://tuoitre.vn/suc-khoe/phong-mach.htm",
        "biet đe khoe": "https://tuoitre.vn/suc-khoe/biet-de-khoe.htm"
    }
}

# Khởi tạo requests session với cơ chế tự động thử lại (Retry)
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

def extract_year(date_str):
    if not date_str: 
        return None
    match = re.search(r'\b(20\d{2})\b', date_str)
    return int(match.group(1)) if match else None

def get_article_details(article_url):    
    """Truy xuất trang chi tiết bằng Requests để tối ưu tốc độ"""
    details = {
        "public_date": "", "content": "", "author": "", 
        "tag": "", "description": "", "source": "Tuổi Trẻ"
    }
    
    if not article_url.startswith("http"):
        article_url = "https://tuoitre.vn" + article_url
        
    try:
        r = session.get(article_url, timeout=10)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "html.parser")
            
            meta_date = soup.find("meta", property="article:published_time")
            if meta_date: details["public_date"] = meta_date.get("content", "")
            
            # --- LẤY TÊN TÁC GIẢ THẬT ĐÚNG CHUẨN ---
            real_author_tag = soup.select_one(".author-info .name")
            if real_author_tag:
                details["author"] = real_author_tag.get_text(strip=True)
            else:
                meta_dable_author = soup.find("meta", property="dable:author")
                if meta_dable_author:
                    details["author"] = meta_dable_author.get("content", "").strip()
                else:
                    meta_author = soup.find("meta", attrs={"name": "author"})
                    if meta_author: details["author"] = meta_author.get("content", "").strip()
            # ----------------------------------------
                
            meta_keywords = soup.find("meta", attrs={"name": "keywords"})
            if meta_keywords: details["tag"] = meta_keywords.get("content", "").strip()

            meta_desc = soup.find("meta", attrs={"name": "description"})
            if meta_desc:
                details["description"] = meta_desc.get("content", "").strip()
            else:
                sapo_tag = soup.select_one(".detail-sapo")
                if sapo_tag: details["description"] = sapo_tag.get_text(strip=True)

            paragraphs = soup.select(".detail-content p")
            details["content"] = "\n".join([p.get_text(strip=True) for p in paragraphs])
                
    except Exception as e:
        print(f"      [!] Lỗi tải chi tiết: {article_url} - {e}")
    return details

def setup_driver():    
    """Khởi tạo Selenium Chrome tương thích 100% với môi trường Kaggle/Docker"""
    chrome_options = Options()
    
    # Các arguments BẮT BUỘC để chạy Selenium trên Kaggle
    chrome_options.add_argument("--headless=new") 
    chrome_options.add_argument("--no-sandbox") 
    chrome_options.add_argument("--disable-dev-shm-usage") 
    chrome_options.add_argument("--window-size=1920,1080")
    
    # Tùy chọn tăng tốc và tránh bị chặn
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--mute-audio")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Chặn tải hình ảnh để tăng tốc độ load trang
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def crawl_topic_with_selenium(driver, major_category, sub_topic, url, target_years, output_filepath):
    data_count = 0
    processed_urls = set()
    consecutive_old_articles = 0
    MAX_OLD_ARTICLES = 25 # Tăng giới hạn để tránh bị dừng sớm do vài bài báo cũ xen kẽ
    min_year = min(target_years)

    print(f"  -> Đang mở {sub_topic}: {url}")
    driver.get(url)
    time.sleep(4) 

    click_count = 0
    
    while True:
        # 1. CUỘN XUỐNG CUỐI TRANG ĐỂ NÚT XEM THÊM XUẤT HIỆN
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight - 1000);")
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        all_articles = soup.select(".box-category-item, .item-first")
        articles = [art for art in all_articles if not art.find_parent(class_=["list-box-right", "box-right"])]
        
        if not articles:
            print("    [!] Không tìm thấy bài viết nào trên trang.")
            break

        current_loop_should_stop = False

        for art in articles:
            title_tag = art.select_one("h3.box-title-text a, a.box-category-link-title")
            if not title_tag: continue

            article_link = title_tag.get("href")
            if not article_link or article_link in processed_urls:
                continue

            processed_urls.add(article_link)
            
            title = title_tag.get_text(strip=True)
            art_details = get_article_details(article_link)
            year = extract_year(art_details["public_date"])

            if year in target_years:
                article_data = {
                    "category": major_category, "sub_topic": sub_topic, "title": title,
                    "description": art_details["description"], "content": art_details["content"],
                    "public_date": art_details["public_date"], "author": art_details["author"],
                    "source": art_details["source"],
                    "url": "https://tuoitre.vn" + article_link if not article_link.startswith("http") else article_link,
                    "tag": art_details["tag"]
                }
                
                # --- CƠ CHẾ LƯU NGAY VÀO FILE CSV ---
                file_exists = os.path.isfile(output_filepath)
                df_row = pd.DataFrame([article_data])
                # mode='a' là Append, header=not file_exists để tự động tạo Header nếu file mới tinh
                df_row.to_csv(output_filepath, mode='a', header=not file_exists, index=False, encoding="utf-8-sig")
                # ------------------------------------
                
                data_count += 1
                consecutive_old_articles = 0
                
                # Log ngắn gọn để theo dõi tiến độ
                if data_count % 5 == 0:
                    print(f"      + Đã thu thập và LƯU THÀNH CÔNG {data_count} bài...")
                
            elif year and year < min_year:
                consecutive_old_articles += 1
                if consecutive_old_articles >= MAX_OLD_ARTICLES:
                    print(f"    [!] Đã đào tới mốc bài cũ ({year}). Hoàn thành chuyên mục.")
                    current_loop_should_stop = True
                    break
        
        if current_loop_should_stop: break

        # 2. XỬ LÝ CLICK "XEM THÊM" BẰNG JAVASCRIPT ÉP BUỘC
        try:
            view_more_selectors = [
                "a.view-more", 
                ".box-viewmore a", 
                "[data-role='viewmore']",
                "a.view-more-chuyenmuc"
            ]
            
            btn_found = False
            for selector in view_more_selectors:
                try:
                    wait = WebDriverWait(driver, 5)
                    view_more_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});", view_more_btn)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", view_more_btn)
                    btn_found = True
                    break
                except:
                    continue
            
            if btn_found:
                click_count += 1
                time.sleep(4) 
            else:
                print(f"    [!] Không tìm thấy nút Xem thêm sau {click_count} lần click. Dừng chuyên mục.")
                break
                
        except Exception as e:
            print(f"    [!] Lỗi khi cố gắng tải thêm bài: {e}")
            break

    return data_count

def main():
    grand_total = 0 
    print("="*60)
    print(f"BẮT ĐẦU CÀO DỮ LIỆU TUỔI TRẺ - NĂM {TARGET_YEARS}")
    print("="*60)

    # Đảm bảo thư mục output tồn tại
    output_dir = "/kaggle/working/"
    os.makedirs(output_dir, exist_ok=True)

    driver = setup_driver()

    try:
        for major_category, sub_categories in CATEGORIES_DATA.items():
            if not sub_categories: continue
            print(f"\n[CHUYÊN MỤC]: {major_category.upper()}")
            
            # File được tạo ra riêng cho từng danh mục lớn
            filename = os.path.join(output_dir, f"TuoiTre_{major_category.replace(' ', '_')}.csv") 

            for sub_topic, url in sub_categories.items():
                # Truyền filename vào để nó lưu trực tiếp bên trong
                topic_count = crawl_topic_with_selenium(driver, major_category, sub_topic, url, TARGET_YEARS, filename)
                grand_total += topic_count
                
            print(f"=> [HOÀN TẤT CHUYÊN MỤC] Dữ liệu đã được lưu an toàn tại: {filename}")

    finally:
        driver.quit()

    print(f"\n{'='*60}")
    print(f"CÀO XONG TOÀN BỘ! Tổng số lượng: {grand_total} bài báo đã lưu trong ổ đĩa.")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

BẮT ĐẦU CÀO DỮ LIỆU TUỔI TRẺ - NĂM [2023, 2024, 2025]

[CHUYÊN MỤC]: THẾ GIỚI
  -> Đang mở binh luan: https://tuoitre.vn/the-gioi/binh-luan.htm
      + Đã thu thập và LƯU THÀNH CÔNG 5 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 10 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 15 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 20 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 25 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 30 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 35 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 40 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 45 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 50 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 55 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 60 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 65 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 70 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 75 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 80 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 85 bài...
      + Đã thu thập và LƯU THÀNH CÔNG 90 